<a href="https://colab.research.google.com/github/jottanovaes/colab-notebooks/blob/main/trabalho_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problema 6 - Escala de funcionários
Uma empresa possui uma equipe que trabalha sete dias por semana em turnos de trabalho de
mesmo tempo.

A equipe possui *n* funcionários. Cada funcionário deve ter 5 folgas durante o
mês e para garantir que o trabalho seja realizado de forma competente todos os dias, a equipe
não pode operar com menos que *l* funcionários.

Por razões contratuais nenhum funcionário pode
trabalhar por 6 dias seguidos sem folgas e ao menos uma das folgas deve ocorrer em um domingo
ou em um feriado. Os funcionários são consultados e indicam datas preferenciais para folgas.
Construa um modelo de otimização que determine a escala dos funcionários para o mês de junho
de 2026 que:

1.   Maximize o número total de folgas nos dias requisitados pelos funcionários.
2.   Distribua as folgas nos dias requisitados de maneira mais equilibrada possível entre os funcionários.

# 1. Maximize o número total de folgas nos dias requisitados pelos funcionários.

## modelo algebrico
- *n* funcionarios
- *m* dia do mês

## variaveis de decisao
$$
x_{ij} =
\begin{cases}
1, & \text{se o funcionário folga no dia} \\
0, & \text{caso contrário}
\end{cases}
$$

## parametros
$$ l = \text{minimo de funcionarios em cada turno} $$
$$ D = \text{domingos e feriados} $$
$$ P_{ij} = \text{preferencia do funcionario } i \text{ para folgar no dia } j $$

## funcao objetivo
$$
 max  \sum_{i=1}^{n}\sum_{j=1}^{m} P_{ij} \cdot x_{ij}
$$

## retricoes
$$
\sum_{j=1}^{m} x_{ij} = 5, \quad \forall i \in \{1, \ldots, n\} \\
\sum_{i=1}^{n} (1 - x_{ij}) \ge l, \quad \forall j \in \{1, \ldots, m\} \\
\sum_{j \in D} x_{ij} \ge 1, \quad \forall i \in \{1, \ldots, n\} \\
\sum_{k=j}^{j+5} x_{ik} \ge 1, \quad \forall i \in \{1,\ldots,n\},\ \forall j \in \{1, \ldots, m-5\} \\
x_{ij} \in \{0, 1\}, \quad \forall i, \forall j
$$

In [ ]:
!pip install gurobipy

In [ ]:
# calendario junho/2026
D = {3, 6, 13, 20, 27} # indice dos dias que sao domingos o feriados
m = 30

In [ ]:
# dados de entrada
n = 20 # numero de funcionarios
l = 15 # numero minimo de funcionarios em cada turno

P = [
    [12, 16, 17, 30],
    [11, 21, 22, 27],
    [7, 8, 20, 24],
    [6, 11, 18, 29],
    [1, 12, 21, 26],
    [20, 21, 23, 27],
    [9, 12, 21, 24],
    [3, 19, 23, 30],
    [11, 15, 18, 20],
    [17, 18, 19, 23],
    [3, 9, 14, 29],
    [4, 8, 13, 25],
    [1, 5, 6, 9],
    [2, 13, 25, 27],
    [18, 22, 23, 26],
    [2, 5, 23, 30],
    [4, 19, 21, 22],
    [12, 22, 27, 29],
    [3, 22, 25, 30],
    [12, 19, 23, 29],
] # preferencias

# converte P para base 0
P = [[d - 1 for d in pref] for pref in P]

# cria matriz binária n x m, 1 se o funcionario i quer folgar no dia j
P_matrix = [
    [1 if j in P[i] else 0 for j in range(m)]
    for i in range(n)
]

In [ ]:
import gurobipy as gp

modelo = gp.Model()

x = modelo.addVars(
    n,
    m,
    vtype = gp.GRB.BINARY,
    name = "x"
)

modelo.setObjective(
    gp.quicksum(P_matrix[i][j] * x[i, j] for i in range(n) for j in range(m)),
    gp.GRB.MAXIMIZE
)

# r1: cada funcionario tem exatamente 5 folgas no mes
r1 = modelo.addConstrs(
    gp.quicksum(x[i, j] for j in range(m)) == 5
    for i in range(n)
)

# r2: minimo de l funcionarios trabalhando por dia
r2 = modelo.addConstrs(
    gp.quicksum(1 - x[i, j] for i in range(n)) >= l
    for j in range(m)
)

# r3: pelo menos uma folga em domingo ou feriado
r3 = modelo.addConstrs(
    gp.quicksum(x[i, j] for j in D) >= 1
    for i in range(n)
)

# r4: nenhum funcionario trabalha 6 dias seguidos sem folga
r4 = modelo.addConstrs(
    gp.quicksum(x[i, k] for k in range(j, j + 6)) >= 1
    for i in range(n)
    for j in range(m - 5)
)

modelo.setParam("OutputFlag", 0)
modelo.optimize()

if modelo.status == gp.GRB.OPTIMAL:
    print(f"Valor otimo: {modelo.objVal:.0f} folgas em dias preferidos\n")

    for i in range(n):
        folgas = [j + 1 for j in range(m) if round(x[i, j].x) == 1]
        preferidas = [j + 1 for j in range(m) if round(x[i, j].x) == 1 and j in P[i]]
        print(f"Funcionário {i+1:2d} | folgas: {folgas} | preferidas atendidas: {preferidas}")
else:
    print("Nao ha solucao otima")

Valor otimo: 46 folgas em dias preferidos

Funcionário  1 | folgas: [6, 12, 17, 22, 28] | preferidas atendidas: [12, 17]
Funcionário  2 | folgas: [5, 11, 15, 21, 27] | preferidas atendidas: [11, 21, 27]
Funcionário  3 | folgas: [2, 8, 14, 20, 26] | preferidas atendidas: [8, 20]
Funcionário  4 | folgas: [6, 12, 18, 23, 28] | preferidas atendidas: [6, 18]
Funcionário  5 | folgas: [6, 12, 15, 21, 26] | preferidas atendidas: [12, 21, 26]
Funcionário  6 | folgas: [4, 10, 16, 21, 27] | preferidas atendidas: [21, 27]
Funcionário  7 | folgas: [6, 12, 15, 21, 27] | preferidas atendidas: [12, 21]
Funcionário  8 | folgas: [3, 8, 14, 19, 25] | preferidas atendidas: [3, 19]
Funcionário  9 | folgas: [5, 11, 14, 20, 26] | preferidas atendidas: [11, 20]
Funcionário 10 | folgas: [5, 11, 17, 23, 28] | preferidas atendidas: [17, 23]
Funcionário 11 | folgas: [3, 9, 14, 19, 25] | preferidas atendidas: [3, 9, 14]
Funcionário 12 | folgas: [4, 8, 13, 19, 25] | preferidas atendidas: [4, 8, 13, 25]
Funcionário 

# 2. Distribua as folgas nos dias requisitados de maneira mais equilibrada possível entre os funcionários.